In [1]:
from datasets import load_from_disk
from openai import AzureOpenAI


/media/research/yrl_aida_users/poddubny/poddubnyy/postgraduate/semtab_serializer/.myenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
import sys
import os

# Добавляем корневую директорию проекта в sys.path
sys.path.append(os.path.dirname(os.path.abspath('/media/research/yrl_aida_users/poddubny/poddubnyy/postgraduate/semtab_serializer/tests')))

In [4]:
dataset = load_from_disk('none_filtered_new_dataset/')
dataset

Dataset({
    features: ['id', 'statement', 'label', 'table_caption', 'table_text', 'pandas_code', 'pandas_eval', 'nlsep_query', 'semtab_query'],
    num_rows: 88116
})

In [42]:
dataset[0]

NameError: name 'dataset' is not defined

In [5]:

system_prompt = '''You are a Python expert specializing in pandas. Your task is to translate the
given natural language statement into a single-line pandas expression. This
expression must be valid and executable to verify the truth of the statement
using the provided table. Consider the following:
1. The table is represented as a pandas DataFrame named df.
2. Do not include explanations, comments, or multiline outputs.
3. Ensure the output is concise, correct, and when run outputs either True or
False, and strictly in the following Json Format with a single key "PANDA":
"PANDA": "<your Pandas code>"
'''


In [6]:
print(system_prompt)

You are a Python expert specializing in pandas. Your task is to translate the
given natural language statement into a single-line pandas expression. This
expression must be valid and executable to verify the truth of the statement
using the provided table. Consider the following:
1. The table is represented as a pandas DataFrame named df.
2. Do not include explanations, comments, or multiline outputs.
3. Ensure the output is concise, correct, and when run outputs either True or
False, and strictly in the following Json Format with a single key "PANDA":
"PANDA": "<your Pandas code>"



In [7]:
from openai import OpenAI
def send_message(message,max_tokens=2000,top_p=0.9,temperature=0.5,server_url="http://127.0.0.1:8800/v1",api_key="dummy",
                 model_name='deepseek-ai/deepseek-coder-7b-instruct-v1.5',system_prompt='', stop = ["Observation:","\n\n\n\n","\n \n \n"]):
    client = OpenAI(base_url=server_url, api_key=api_key)
    model_input = [
        { 'role': 'system', 'content': system_prompt},
        { 'role': 'user', 'content': message}
    ]
    try:
        print(f"Generating content with model: {model_name}",)
        
        response = client.chat.completions.create(
            model=model_name,
            messages=model_input,
            temperature=temperature,
            max_tokens=max_tokens,
            top_p=top_p,
            stop = stop
        )
        
        return True, response.choices[0].message.content

    except Exception as e:
        print("Failed to call LLM: " + str(e))
        time.sleep(6)
        if hasattr(e, 'response'):
            error_info = e.response.json()  
            code_value = error_info['error']['code']
        else:
            code_value = "context_length_exceeded"
        print("Retrying ...")
        return False, None

In [8]:
import json
import pandas as pd
from io import StringIO
from tqdm import tqdm

In [ ]:

count = 0
corrects_sep = 0
corrects_sem = 0
for entry in tqdm(dataset):
    total += 1
    df = pd.read_csv(StringIO(entry['table_text']), delimiter='#')
    success,response_sep = send_message(f'{entry["statement"]} {entry["nlsep_query"]}',
                                       system_prompt=system_prompt)
    if success:
        pandas_eval = str(bool(eval(response_sep)))
        if str(bool(entry['label'])) == str(pandas_eval):
            corrects_sep += 1
        else:
            print(response_sep)

    success,response_sem = send_message(f'{entry["statement"]} {entry["semtab_query"]}',
                                       system_prompt=system_prompt)
    if success:
        pandas_eval = str(bool(eval(response_sem)))
        if str(bool(entry['label'])) == str(pandas_eval):
            response_sem += 1
        else:
            print(corrects_sem)

print(corrects, total, corrects / total)

In [76]:
import re

def parse_panda_code(input_string):
    """
    Парсит строку и извлекает код, который находится внутри конструкции "PANDA": <код>
    Поддерживает различные форматы: JSON, простой текст, Markdown
    
    Args:
        input_string (str): Входная строка для парсинга
        
    Returns:
        str: Извлеченный код или пустая строка, если код не найден
    """
    # Сначала попробуем найти JSON объект с PANDA
    json_pattern = r'\{[^{}]*PANDA":\s*(.+?)(?:\n|$)?\}'
    json_match = re.search(json_pattern, input_string, re.DOTALL)
    code = None
    pattern = r'"PANDA":\s*(.+?)(?:\n|$)'
    if json_match:
        code = json_match.group(1).strip()
    else:
        match = re.search(pattern, input_string, re.DOTALL)
        if match:
        # Извлекаем код и убираем лишние пробелы по краям
            code = match.group(1).strip()
        # Заменяем одинарные кавычки внутри строки для корректного парсинга JSON

    # Если JSON не найден или не распарсился, используем старый метод
    # Паттерн для поиска кода после "PANDA": 
    # Ищет "PANDA": за которым следует пробел, затем код до конца строки или до следующего символа
    if code != None:
    # Убираем возможные кавычки вокруг кода
        if code.startswith('"') and code.endswith('"'):
            code = code[1:-1]
        elif code.startswith("'") and code.endswith("'"):
            code = code[1:-1]
            
        return code
    
    return ""

In [81]:
corrects_sep = 0

df = pd.read_csv(StringIO(dataset[0]['table_text']), delimiter='#')
success,response_sep = send_message(f'{dataset[0]["statement"]} {dataset[0]["nlsep_query"]}',
                                   system_prompt=system_prompt)
if success:
    pandas_eval = str(bool(eval(parse_panda_code(response_sep))))
    if str(bool(dataset[0]['label'])) == str(pandas_eval):
        corrects_sep += 1
    else:
        print(response_sep)

success,response_sem = send_message(f'{dataset[0]["statement"]} {dataset[0]["semtab_query"]}',
                                   system_prompt=system_prompt)

Generating content with model: deepseek-ai/deepseek-coder-7b-instruct-v1.5
Here is a single-line pandas expression that checks if Haroldo is mentioned as a Brazil scorer for two different games:

```json
"PANDA": "df['brazil scorers'].apply(lambda x: 'haroldo' in x.split(', ')).sum() >= 2"
```

This expression splits the 'brazil scorers' column into a list of scorers for each game, checks if 'haroldo' is in the list, and then sums up the True values to see if he is mentioned as a scorer for at least two games.

Generating content with model: deepseek-ai/deepseek-coder-7b-instruct-v1.5


In [42]:
print(response_sep)

```json
"PANDA": "df['brazil scorers'].str.contains('haroldo', na=False).sum() > 1"
```
This code checks if 'haroldo' is mentioned as a Brazil scorer for more than 1 game. It uses the str.contains method to check if 'haroldo' is present in the 'brazil scorers' column and then sums up the number of True values, indicating how many games 'haroldo' scored for Brazil.



In [43]:
dataset[0]['label']

1

In [44]:
str(bool(eval(parse_panda_code(response_sep))))

'True'

In [45]:
parse_panda_code(response_sep)

"df['brazil scorers'].str.contains('haroldo', na=False).sum() > 1"

In [90]:
df['brazil scorers'].apply(lambda x: 'haroldo' in x.split(', ')).sum() >= 2

np.False_

In [82]:
print(response_sem)

```json
"PANDA": "df['brazil scorers'].apply(lambda x: 'haroldo' in x.split(',')).sum() > 1"
```
This code checks if 'haroldo' is mentioned as a scorer for more than one game in the 'brazil scorers' column of the DataFrame.



In [83]:
corrects_sem = 0
if success:
    pandas_eval = str(bool(eval(parse_panda_code(response_sem))))
    if str(bool(dataset[0]['label'])) == str(parse_panda_code(pandas_eval)):
        corrects_sem += 1
    else:
        print(corrects_sem)

0


In [77]:
parse_panda_code(response_sem)

"df['brazil scorers'].str.contains('haroldo', na=False).sum() > 1"

In [65]:
response_sem

'Here is the single-line pandas expression based on the given statement:\n\n```json\n{"PANDA": "df[\'brazil scorers\'].str.contains(\'haroldo\', na=False).sum() > 1"}\n```\n\nThis expression checks if \'haroldo\' is mentioned as a Brazil scorer for more than 1 game in the \'brazil scorers\' column of the DataFrame df.\n'

In [66]:
pattern = r'"PANDA":\s*(.+?)(?:\n|$)'
    
    # Используем поиск с флагом re.DOTALL для работы с многострочным кодом
match = re.search(pattern, '''Generating content with model: deepseek-ai/deepseek-coder-7b-instruct-v1.5
Here is a Python pandas code snippet that checks if Haroldo is mentioned as a Brazil scorer for more than one game:
```json
"PANDA": "df['brazil scorers'].str.contains('Haroldo', na=False).sum() > 1"
```
This code uses the pandas `str.contains` method to check if 'Haroldo' is in the 'brazil scorers' column. The `na=False` parameter ensures that NaN values are not considered as matches. The `sum()` method is then used to count the number of True values returned by the `contains` method, which corresponds to the number of times 'Haroldo' is mentioned as a Brazil scorer. The result is checked if it's greater than 1.''',
                  re.DOTALL)

In [67]:
code = match.group(1).strip()

In [68]:
code

'"df[\'brazil scorers\'].str.contains(\'Haroldo\', na=False).sum() > 1"'

In [73]:
match.group()

'"PANDA": "df[\'brazil scorers\'].str.contains(\'Haroldo\', na=False).sum() > 1"\n'

In [85]:
bool(eval(parse_panda_code(response_sem)))

False

In [86]:
bool(dataset[0]['label'])

True

In [167]:
from datasets import load_from_disk
from openai import OpenAI
import sys
import os
import json
import pandas as pd
from io import StringIO
from tqdm import tqdm
import re
# Добавляем корневую директорию проекта в sys.path
sys.path.append(os.path.dirname(os.path.abspath('/media/research/yrl_aida_users/poddubny/poddubnyy/postgraduate/semtab_serializer/tests')))

system_prompt = '''You are a Python expert specializing in pandas. Your task is to translate the
given natural language statement into a single-line pandas expression. This
expression must be valid and executable to verify the truth of the statement
using the provided table. Consider the following:
1. The table is represented as a pandas DataFrame named df.
2. Do not include explanations, comments, or multiline outputs.
3. Ensure the output is concise, correct, and when run outputs either True or
False, and strictly in the following Json Format with a single key "PANDA":
"PANDA": "<your Pandas code>"
'''
system_correcting_prompt = '''
You are a Python expert specializing in pandas. Your task is to correct a pandas code that translates 
a given natural language statement into a pandas
expression. The input data, code, along with the specific error it contains, is provided.
Your corrected pandas_code must be valid and executable by running the code
snippet str(bool(eval(pandas_code))) ensuring it accurately evaluates the truth
of the statement using the provided table with no errors.
Make sure the pandas_code is of type boolean. Consider the following:
1. The table is represented as a pandas DataFrame named df.
2. Do not include explanations, comments, or multiline outputs.
3. Ensure the output is concise, correct, and when run outputs either True or
False, and strictly in the following Json Format with a single key "CORRECT PANDA": 
"CORRECT PANDA": "<your Pandas code>"
'''


def send_message(message,max_tokens=500,top_p=0.9,temperature=0.5,server_url="http://127.0.0.1:8800/v1",api_key="dummy",
                 model_name='deepseek-ai/deepseek-coder-7b-instruct-v1.5',system_prompt='', stop = ["Observation:","\n\n\n\n","\n \n \n"]):
    client = OpenAI(base_url=server_url, api_key=api_key)
    model_input = [
        { 'role': 'system', 'content': system_prompt},
        { 'role': 'user', 'content': message}
    ]
    try:
        print(f"Generating content with model: {model_name}",)
        
        response = client.chat.completions.create(
            model=model_name,
            messages=model_input,
            temperature=temperature,
            max_tokens=max_tokens,
            top_p=top_p,
            stop = stop
        )
        
        return True, response.choices[0].message.content

    except Exception as e:
        print("Failed to call LLM: " + str(e))
        time.sleep(6)
        if hasattr(e, 'response'):
            error_info = e.response.json()  
            code_value = error_info['error']['code']
        else:
            code_value = "context_length_exceeded"
        print("Retrying ...")
        return False, None

def parse_panda_code(input_string):
    """
    Парсит строку и извлекает код, который находится внутри конструкции "PANDA": <код>
    Поддерживает различные форматы: JSON, простой текст, Markdown
    
    Args:
        input_string (str): Входная строка для парсинга
        
    Returns:
        str: Извлеченный код или пустая строка, если код не найден
    """
    # Сначала попробуем найти JSON объект с PANDA
    json_pattern = r'\{[^{}]*PANDA":\s*(.+?)(?:\n|$)?\}'
    json_match = re.search(json_pattern, input_string, re.DOTALL)
    code = None
    pattern = r'"PANDA":\s*(.+?)(?:\n|$)'
    if json_match:
        code = json_match.group(1).strip()
    else:
        match = re.search(pattern, input_string, re.DOTALL)
        if match:
        # Извлекаем код и убираем лишние пробелы по краям
            code = match.group(1).strip()
        # Заменяем одинарные кавычки внутри строки для корректного парсинга JSON

    # Если JSON не найден или не распарсился, используем старый метод
    # Паттерн для поиска кода после "PANDA": 
    # Ищет "PANDA": за которым следует пробел, затем код до конца строки или до следующего символа
    if code != None:
    # Убираем возможные кавычки вокруг кода
        if code.startswith('"') and code.endswith('"'):
            code = code[1:-1]
        elif code.startswith("'") and code.endswith("'"):
            code = code[1:-1]
            
        return code
    
    return ""

def correct_code(df,input_data,code_str,error_str,iter_id=0,system_correcting_prompt='',stop= ["Observation:","\n\n\n\n","\n \n \n"],
                max_tokens=500,top_p=0.9,temperature=0.5,server_url="http://127.0.0.1:8800/v1",api_key="dummy",
                model_name='deepseek-ai/deepseek-coder-7b-instruct-v1.5',max_iter=3):
    if iter_id<max_iter:
        success,response = send_message(f'INPUT DATA: {input_data}\n CODE: {code_str}\n ERROR: {error_str}',stop= stop,
                    max_tokens=max_tokens,top_p=top_p,temperature=temperature,server_url=server_url,api_key=api_key,
                    model_name=model_name, system_prompt=system_correcting_prompt)
        if success:
            correct_code = parse_panda_code(response_sep)
            try:
                pandas_eval = str(bool(eval(correct_code)))
                return True,correct_code, response
            except Exception as e:
                print(e)
                return correct_code(df,correct_code,str(e),iter_id=iter_id+1,max_tokens=max_tokens,top_p=top_p,
                             temperature=temperature, server_url=server_url,api_key=api_key,model_name=model_name, 
                             system_correcting_prompt=system_correcting_prompt,max_iter=max_iter)
                
        else:
            print('success correcting error')
            return correct_code(df,correct_code,str(e),iter_id=iter_id+1,max_tokens=max_tokens,top_p=top_p,
                             temperature=temperature, server_url=server_url,api_key=api_key,model_name=model_name, 
                             system_correcting_prompt=system_correcting_prompt,max_iter=max_iter)
    else:
        print(f'end of iter {iter_id}, max iter is {max_iter}')
        return False, code_str, None

system_correcting_prompt
def dataset_processing(entry,query_field=None,system_correcting_prompt='',system_prompt=''):
    df = pd.read_csv(StringIO(entry['table_text']), delimiter='#')
    success,response_sep = send_message(entry[f"{query_field}_query"],
                                       system_prompt=system_prompt)

    entry[f'{query_field}_answ'] = 'None'
    entry[f'{query_field}_label'] = 'None'
    entry[f'{query_field}_answ_correct'] = 'None'

    
    if success:
        entry[f'{query_field}_answ'] = response_sep
        try:
            code = parse_panda_code(response_sep)
            try:
            
                pandas_eval = str(bool(eval(code)))
                entry[f'{query_field}_label'] = str(pandas_eval)
            except Exception as e:
                print('EEEERRRRRRR', entry['id'])
                print(e)
                success_correcting, new_code, correcting_response = correct_code(df,entry[f"{query_field}_query"],
                                             correct_code,str(e),
                                             system_correcting_prompt=system_correcting_prompt)
                if success_correcting:
                    pandas_eval = str(bool(eval(new_code)))
                    entry[f'{query_field}_label'] = str(pandas_eval)
                    entry[f'{query_field}_answ_correct'] = correcting_response
        except Exception as e:
            print (e)
        
    return entry

#def dataset_processing(entry):
#    df = pd.read_csv(StringIO(entry['table_text']), delimiter='#')
#    success,response_sep = send_message(f'{entry["statement"]} {entry["nlsep_query"]}',
#                                       system_prompt=system_prompt)
#    if success:
#        entry['sep_answ'] = response_sep
#        pandas_eval = str(bool(eval(parse_panda_code(response_sep))))
#        entry['sep_label'] = str(pandas_eval)
#    else:
#        entry['sep_answ'] = 'None'
#        entry['sep_label'] = 'None'
#        
#    success,response_sem = send_message(f'{entry["statement"]} {entry["semtab_query"]}',
#                                       system_prompt=system_prompt)
#   if success:
#       entry['sem_answ'] = response_sem
#        pandas_eval = str(bool(eval(parse_panda_code(response_sem))))
#        entry['sem_label'] = str(pandas_eval)
#    else:
#        entry['sem_answ'] = 'None'
#        entry['sem_label'] = 'None'
#dataset = load_from_disk('none_filtered_new_dataset/')
#dataset2 = dataset.map(dataset_processing)


In [97]:
type(success)

bool

In [124]:
process_data = load_from_disk('answer_gen_semtab_none_filtered_new_dataset')
process_data

Dataset({
    features: ['id', 'statement', 'label', 'table_caption', 'table_text', 'pandas_code', 'pandas_eval', 'nlsep_query', 'semtab_query', 'semtab_answ', 'semtab_label'],
    num_rows: 88116
})

In [127]:
dd = process_data.filter(lambda x: True if x['semtab_answ']!='None' else False,num_proc=17)
dd

Dataset({
    features: ['id', 'statement', 'label', 'table_caption', 'table_text', 'pandas_code', 'pandas_eval', 'nlsep_query', 'semtab_query', 'semtab_answ', 'semtab_label'],
    num_rows: 88116
})

In [128]:
dd = process_data.filter(lambda x: True if x['semtab_label']!='None' else False,num_proc=17)
dd

Filter (num_proc=17): 100%|██████████| 88116/88116 [00:00<00:00, 151805.97 examples/s]


Dataset({
    features: ['id', 'statement', 'label', 'table_caption', 'table_text', 'pandas_code', 'pandas_eval', 'nlsep_query', 'semtab_query', 'semtab_answ', 'semtab_label'],
    num_rows: 62012
})

In [129]:
dd.shape[0]/process_data.shape[0]*100

70.37541422670117

In [130]:
true = dd.filter(lambda x : True if x['semtab_label']==str(bool(x['label'])) else False,num_proc=17)

Filter (num_proc=17): 100%|██████████| 62012/62012 [00:00<00:00, 98299.48 examples/s]


In [132]:
true.shape[0]/process_data.shape[0]*100


40.90176585410141

In [131]:
true.shape[0]/dd.shape[0]*100


58.11939624588789

In [134]:
process_data = load_from_disk('answer_gen_nlsep_none_filtered_new_dataset/')
process_data

Dataset({
    features: ['id', 'statement', 'label', 'table_caption', 'table_text', 'pandas_code', 'pandas_eval', 'nlsep_query', 'semtab_query', 'nlsep_answ', 'nlsep_label'],
    num_rows: 88116
})

In [135]:
dd = process_data.filter(lambda x: True if x['nlsep_answ']!='None' else False,num_proc=17)
dd

Filter (num_proc=17): 100%|██████████| 88116/88116 [00:00<00:00, 156255.01 examples/s]


Dataset({
    features: ['id', 'statement', 'label', 'table_caption', 'table_text', 'pandas_code', 'pandas_eval', 'nlsep_query', 'semtab_query', 'nlsep_answ', 'nlsep_label'],
    num_rows: 88007
})

In [136]:
dd = process_data.filter(lambda x: True if x['nlsep_label']!='None' else False,num_proc=17)
dd

Filter (num_proc=17): 100%|██████████| 88116/88116 [00:00<00:00, 162598.23 examples/s]


Dataset({
    features: ['id', 'statement', 'label', 'table_caption', 'table_text', 'pandas_code', 'pandas_eval', 'nlsep_query', 'semtab_query', 'nlsep_answ', 'nlsep_label'],
    num_rows: 59619
})

In [137]:
dd.shape[0]/process_data.shape[0]*100

67.65967588179218

In [139]:
true = dd.filter(lambda x : True if x['nlsep_label']==str(bool(x['label'])) else False,num_proc=17)

Filter (num_proc=17): 100%|██████████| 59619/59619 [00:00<00:00, 99274.44 examples/s] 


In [140]:
true.shape[0]/dd.shape[0]*100


62.41298914775491

In [141]:
true.shape[0]/process_data.shape[0]*100


42.228426165509106

In [150]:
process_data = load_from_disk('answer_gen_proto_none_filtered_new_dataset/')
process_data

Dataset({
    features: ['id', 'statement', 'label', 'table_caption', 'table_text', 'pandas_code', 'pandas_eval', 'nlsep_query', 'semtab_query', 'proto_query', 'proto_answ', 'proto_label'],
    num_rows: 88116
})

In [151]:
dd = process_data.filter(lambda x: True if x['proto_answ']!='None' else False,num_proc=17)
dd

Dataset({
    features: ['id', 'statement', 'label', 'table_caption', 'table_text', 'pandas_code', 'pandas_eval', 'nlsep_query', 'semtab_query', 'proto_query', 'proto_answ', 'proto_label'],
    num_rows: 88065
})

In [152]:
dd = process_data.filter(lambda x: True if x['proto_label']!='None' else False,num_proc=17)
dd

Dataset({
    features: ['id', 'statement', 'label', 'table_caption', 'table_text', 'pandas_code', 'pandas_eval', 'nlsep_query', 'semtab_query', 'proto_query', 'proto_answ', 'proto_label'],
    num_rows: 46593
})

In [153]:
dd.shape[0]/process_data.shape[0]*100

52.876889554677916

In [154]:
true = dd.filter(lambda x : True if x['proto_label']==str(bool(x['label'])) else False,num_proc=17)

In [155]:
true.shape[0]/process_data.shape[0]*100


31.886377048436152

In [131]:
true.shape[0]/dd.shape[0]*100


58.11939624588789

In [156]:
process_data = load_from_disk('./tab_fact_test_xml_semtab/')
print(process_data)
dd = process_data.filter(lambda x: True if x['semtab_answ']!='None' else False,num_proc=17)
print(dd)
dd = process_data.filter(lambda x: True if x['semtab_label']!='None' else False,num_proc=17)
print(dd)
print(dd.shape[0]/process_data.shape[0]*100)
true = dd.filter(lambda x : True if x['semtab_label']==str(bool(x['label'])) else False,num_proc=17)
print(true.shape[0]/process_data.shape[0]*100)

Dataset({
    features: ['id', 'table_csv', 'table_text', 'label', 'statement', 'table_caption', 'nlsep_query', 'semtab_query', 'semtab_answ', 'semtab_label'],
    num_rows: 12779
})


Filter (num_proc=17): 100%|██████████| 12779/12779 [00:00<00:00, 28116.82 examples/s]


Dataset({
    features: ['id', 'table_csv', 'table_text', 'label', 'statement', 'table_caption', 'nlsep_query', 'semtab_query', 'semtab_answ', 'semtab_label'],
    num_rows: 12779
})


Filter (num_proc=17): 100%|██████████| 12779/12779 [00:00<00:00, 28618.56 examples/s]


Dataset({
    features: ['id', 'table_csv', 'table_text', 'label', 'statement', 'table_caption', 'nlsep_query', 'semtab_query', 'semtab_answ', 'semtab_label'],
    num_rows: 8858
})
69.316847953674


Filter (num_proc=17): 100%|██████████| 8858/8858 [00:00<00:00, 18089.66 examples/s]

40.636982549495265


In [160]:
process_data = load_from_disk('./tab_fact_test_xml_nlsep/')
print(process_data)
dd = process_data.filter(lambda x: True if x['nlsep_answ']!='None' else False,num_proc=17)
print(dd)
dd = process_data.filter(lambda x: True if x['nlsep_label']!='None' else False,num_proc=17)
print(dd)
print(dd.shape[0]/process_data.shape[0]*100)
true = dd.filter(lambda x : True if x['nlsep_label']==str(bool(x['label'])) else False,num_proc=17)
print(true.shape[0]/process_data.shape[0]*100)

Dataset({
    features: ['id', 'table_csv', 'table_text', 'label', 'statement', 'table_caption', 'nlsep_query', 'semtab_query', 'nlsep_answ', 'nlsep_label'],
    num_rows: 12779
})
Dataset({
    features: ['id', 'table_csv', 'table_text', 'label', 'statement', 'table_caption', 'nlsep_query', 'semtab_query', 'nlsep_answ', 'nlsep_label'],
    num_rows: 12779
})
Dataset({
    features: ['id', 'table_csv', 'table_text', 'label', 'statement', 'table_caption', 'nlsep_query', 'semtab_query', 'nlsep_answ', 'nlsep_label'],
    num_rows: 8522
})
66.68753423585571
41.341263009625166


In [161]:
dont_run = process_data.filter(lambda x: True if x['nlsep_label']=='None' else False,num_proc=17)

Filter (num_proc=17): 100%|██████████| 12779/12779 [00:00<00:00, 27883.23 examples/s]


In [162]:
dont_run


Dataset({
    features: ['id', 'table_csv', 'table_text', 'label', 'statement', 'table_caption', 'nlsep_query', 'semtab_query', 'nlsep_answ', 'nlsep_label'],
    num_rows: 4257
})

In [ ]:
from functools import partial
dont_run_correcting = dont_run.map(partial(dataset_processing,query_field='nlsep',system_correcting_prompt=system_correcting_prompt,
                                           system_prompt=system_prompt),num_proc=16)


Map (num_proc=16):   0%|          | 0/4257 [00:00<?, ? examples/s]

Generating content with model: deepseek-ai/deepseek-coder-7b-instruct-v1.5
Generating content with model: deepseek-ai/deepseek-coder-7b-instruct-v1.5
Generating content with model: deepseek-ai/deepseek-coder-7b-instruct-v1.5
Generating content with model: deepseek-ai/deepseek-coder-7b-instruct-v1.5Generating content with model: deepseek-ai/deepseek-coder-7b-instruct-v1.5

Generating content with model: deepseek-ai/deepseek-coder-7b-instruct-v1.5
Generating content with model: deepseek-ai/deepseek-coder-7b-instruct-v1.5
Generating content with model: deepseek-ai/deepseek-coder-7b-instruct-v1.5
Generating content with model: deepseek-ai/deepseek-coder-7b-instruct-v1.5
Generating content with model: deepseek-ai/deepseek-coder-7b-instruct-v1.5Generating content with model: deepseek-ai/deepseek-coder-7b-instruct-v1.5Generating content with model: deepseek-ai/deepseek-coder-7b-instruct-v1.5


Generating content with model: deepseek-ai/deepseek-coder-7b-instruct-v1.5
Generating content with m

Map (num_proc=16):   0%|          | 1/4257 [00:01<1:25:02,  1.20s/ examples]

Generating content with model: deepseek-ai/deepseek-coder-7b-instruct-v1.5
Generating content with model: deepseek-ai/deepseek-coder-7b-instruct-v1.5
Generating content with model: deepseek-ai/deepseek-coder-7b-instruct-v1.5
EEEERRRRRRR EEEERRRRRRR75 
706can't multiply sequence by non-int of type 'float'

The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().
EEEERRRRRRR 880
leading zeros in decimal integer literals are not permitted; use an 0o prefix for octal integers (<string>, line 1)
Generating content with model: deepseek-ai/deepseek-coder-7b-instruct-v1.5
Generating content with model: deepseek-ai/deepseek-coder-7b-instruct-v1.5
Generating content with model: deepseek-ai/deepseek-coder-7b-instruct-v1.5
EEEERRRRRRR 1538
Can only use .dt accessor with datetimelike values


Map (num_proc=16):   0%|          | 3/4257 [00:01<36:14,  1.96 examples/s]  

...

Map (num_proc=16):  27%|██▋       | 1132/4257 [05:14<17:32,  2.97 examples/s]

'brazil scorers'
'str' object is not callable
Generating content with model: deepseek-ai/deepseek-coder-7b-instruct-v1.5
Generating content with model: deepseek-ai/deepseek-coder-7b-instruct-v1.5
'brazil scorers'
'str' object is not callable


Map (num_proc=16):  27%|██▋       | 1134/4257 [05:15<15:24,  3.38 examples/s]

Generating content with model: deepseek-ai/deepseek-coder-7b-instruct-v1.5


Map (num_proc=16):  27%|██▋       | 1135/4257 [05:15<14:24,  3.61 examples/s]

In [208]:
data = load_from_disk('tab_fact_test_xml_nlsep_correct/')

In [209]:
d2 = data.filter(lambda x : x['nlsep_answ_correct'] != 'None',num_proc=17)

In [210]:
d2

Dataset({
    features: ['id', 'table_csv', 'table_text', 'label', 'statement', 'table_caption', 'nlsep_query', 'semtab_query', 'nlsep_answ', 'nlsep_label', 'nlsep_answ_correct', 'nlsep_label_correct'],
    num_rows: 471
})

In [179]:

data[0]

{'id': 0,
 'table_csv': '2-1570274-4.html.csv',
 'table_text': 'tournament#wins#top - 5#top - 10#top - 25#events#cuts made\nmasters tournament#0#1#2#4#4#4\nus open#0#2#3#4#6#5\nthe open championship#1#2#2#2#3#3\npga championship#0#0#1#2#5#4\ntotals#1#5#8#12#18#16\n',
 'label': 1,
 'statement': 'tony lema do not win in the us open',
 'table_caption': 'tony lema',
 'nlsep_query': 'tony lema do not win in the us open  col : tournament | wins | top - 5 | top - 10 | top - 25 | events | cuts made row 1 : masters tournament | 0 | 1 | 2 | 4 | 4 | 4 row 2 : us open | 0 | 2 | 3 | 4 | 6 | 5 row 3 : the open championship | 1 | 2 | 2 | 2 | 3 | 3 row 4 : pga championship | 0 | 0 | 1 | 2 | 5 | 4 row 5 : totals | 1 | 5 | 8 | 12 | 18 | 16',
 'semtab_query': 'tony lema do not win in the us open <TABLE><DESCRIPTION>tony lema</DESCRIPTION><HEADER><NAME>tournament</NAME><SEMANTIC_TYPE>category - 1.0</SEMANTIC_TYPE><DATA_TYPE>"str"</DATA_TYPE><HAS_NONE>0</HAS_NONE><EXAMPLES>["us open", "the open championshi

In [240]:
eval(parse_panda_code(d2[0]['nlsep_answ']))

KeyError: 'tournament'

# Traceback


In [243]:
import traceback

In [248]:
d2[0]

{'id': 0,
 'table_csv': '2-1570274-4.html.csv',
 'table_text': 'tournament#wins#top - 5#top - 10#top - 25#events#cuts made\nmasters tournament#0#1#2#4#4#4\nus open#0#2#3#4#6#5\nthe open championship#1#2#2#2#3#3\npga championship#0#0#1#2#5#4\ntotals#1#5#8#12#18#16\n',
 'label': 0,
 'statement': 'tournament that tony lema have not participate in include the master tournament , the us open , the pga championship and the open championship',
 'table_caption': 'tony lema',
 'nlsep_query': 'tournament that tony lema have not participate in include the master tournament , the us open , the pga championship and the open championship  col : tournament | wins | top - 5 | top - 10 | top - 25 | events | cuts made row 1 : masters tournament | 0 | 1 | 2 | 4 | 4 | 4 row 2 : us open | 0 | 2 | 3 | 4 | 6 | 5 row 3 : the open championship | 1 | 2 | 2 | 2 | 3 | 3 row 4 : pga championship | 0 | 0 | 1 | 2 | 5 | 4 row 5 : totals | 1 | 5 | 8 | 12 | 18 | 16',
 'semtab_query': 'tournament that tony lema have not

In [249]:
df = pd.read_csv(StringIO(d2[0]['table_text']), delimiter='#')
df

,tournament,wins,top - 5,top - 10,top - 25,events,cuts made
0,masters tournament,0,1,2,4,4,4
1,us open,0,2,3,4,6,5
2,the open championship,1,2,2,2,3,3
3,pga championship,0,0,1,2,5,4
4,totals,1,5,8,12,18,16


In [246]:
parse_panda_code(d2[0]['nlsep_answ'])

"df[df['tournament'].isin(['the master tournament', 'the us open', 'the pga championship', 'the open championship']) & (df['participant'] != 'tony lema')].empty"

In [292]:
try:
    p=parse_panda_code(d2[0]['nlsep_answ'])
    eval(p)
except Exception as e:
    print (f'{type(e).__name__}: {e}')
    
    #traceback.print_exc()
    x = traceback.format_exc()
    #print(x)

KeyError: 'participant'


In [301]:
p = df.info

In [318]:
p = df.dtypes

In [319]:
p = p.to_string()

In [322]:
print(p)

tournament    object
wins           int64
top - 5        int64
top - 10       int64
top - 25       int64
events         int64
cuts made      int64


In [194]:
parse_panda_code(d2[2]['nlsep_answ_correct'])

"df['total'].isin([df.loc[df['player'] == 'matías suárez', 'total'].values[0] + 4]).any()"

In [180]:
d2[0]

{'id': 12,
 'table_csv': '2-13135264-6.html.csv',
 'table_text': 'date#visitor#score#home#decision#attendance#record\njanuary 2#detroit#4 - 1#carolina#joseph#17053#24 - 12 - 4 - 1\njanuary 3#anaheim#1 - 3#detroit#legace#20066#25 - 12 - 4 - 1\njanuary 5#nashville#0 - 6#detroit#joseph#20066#26 - 12 - 4 - 1\njanuary 7#boston#3 - 0#detroit#joseph#20066#26 - 13 - 4 - 1\njanuary 10#detroit#1 - 2#boston#joseph#17565#26 - 13 - 4 - 2\njanuary 14#chicago#2 - 4#detroit#legace#20066#27 - 13 - 4 - 2\njanuary 16#phoenix#3 - 3#detroit#joseph#20066#27 - 13 - 5 - 2\njanuary 19#detroit#1 - 2#san jose#joseph#17361#27 - 14 - 5 - 2\njanuary 21#detroit#2 - 2#anaheim#legace#17174#27 - 14 - 6 - 2\njanuary 22#detroit#5 - 4#los angeles#joseph#18118#28 - 14 - 6 - 2\njanuary 24#detroit#2 - 5#phoenix#joseph#19019#28 - 15 - 6 - 2\njanuary 26#detroit#2 - 2#dallas#legace#18532#28 - 15 - 7 - 2\njanuary 29#new jersey#2 - 5#detroit#joseph#20066#29 - 15 - 7 - 2\njanuary 31#carolina#4 - 4#detroit#legace#20066#30 - 15 - 8 

In [196]:
d3 = d2.filter(lambda x : x['nlsep_label'] != 'None',num_proc=17)

Filter (num_proc=17): 100%|██████████| 471/471 [00:00<00:00, 974.02 examples/s]


In [197]:
d3

Dataset({
    features: ['id', 'table_csv', 'table_text', 'label', 'statement', 'table_caption', 'nlsep_query', 'semtab_query', 'nlsep_answ', 'nlsep_label', 'nlsep_answ_correct', 'nlsep_label_correct'],
    num_rows: 0
})

In [200]:
d3 = d2.filter(lambda x : x['nlsep_label_correct']==str(bool(x['label'])),num_proc=17)

Filter (num_proc=17): 100%|██████████| 471/471 [00:00<00:00, 918.59 examples/s]


In [201]:
d3

Dataset({
    features: ['id', 'table_csv', 'table_text', 'label', 'statement', 'table_caption', 'nlsep_query', 'semtab_query', 'nlsep_answ', 'nlsep_label', 'nlsep_answ_correct', 'nlsep_label_correct'],
    num_rows: 296
})

In [205]:
df.dtypes

date              object
result            object
score             object
brazil scorers    object
competition       object
dtype: object

In [204]:
for col in df.columns:
    try:
        df[col] = pd.to_numeric(df[col])
    except ValueError:
        # Если возникает ошибка, оставляем столбец как есть
        pass

In [206]:
df

,date,result,score,brazil scorers,competition
0,"may 11 , 1919",w,6 - 0,"friedenreich (3) , neco (2) , haroldo",south american championship
1,"may 18 , 1919",w,3 - 1,"heitor , amílcar , millon",south american championship
2,"may 26 , 1919",d,2 - 2,neco (2),south american championship
3,"may 29 , 1919",w,1 - 0,friedenreich,south american championship
4,"june 1 , 1919",d,3 - 3,"haroldo , arlindo (2)",taça roberto cherry


In [212]:
data = load_from_disk('tab_fact_test_xml_nlsep_correct_dtype/')
d2 = data.filter(lambda x : x['nlsep_answ_correct'] != 'None',num_proc=17)


Filter (num_proc=17): 100%|██████████| 4257/4257 [00:00<00:00, 8874.34 examples/s]


In [213]:
d2

Dataset({
    features: ['id', 'table_csv', 'table_text', 'label', 'statement', 'table_caption', 'nlsep_query', 'semtab_query', 'nlsep_answ', 'nlsep_label', 'nlsep_answ_correct', 'nlsep_label_correct'],
    num_rows: 415
})

In [214]:
process_data = load_from_disk('./tab_fact_test_xml_nlsep_new_correct//')
print(process_data)
dd = process_data.filter(lambda x: True if x['nlsep_answ']!='None' else False,num_proc=17)
print(dd)
dd = process_data.filter(lambda x: True if x['nlsep_label']!='None' else False,num_proc=17)
print(dd)
print(dd.shape[0]/process_data.shape[0]*100)
true = dd.filter(lambda x : True if x['nlsep_label']==str(bool(x['label'])) else False,num_proc=17)
print(true.shape[0]/process_data.shape[0]*100)

Dataset({
    features: ['id', 'table_csv', 'table_text', 'label', 'statement', 'table_caption', 'nlsep_query', 'semtab_query', 'nlsep_answ', 'nlsep_label', 'nlsep_answ_correct'],
    num_rows: 12779
})


Filter (num_proc=17): 100%|██████████| 12779/12779 [00:00<00:00, 27151.13 examples/s]


Dataset({
    features: ['id', 'table_csv', 'table_text', 'label', 'statement', 'table_caption', 'nlsep_query', 'semtab_query', 'nlsep_answ', 'nlsep_label', 'nlsep_answ_correct'],
    num_rows: 12779
})


Filter (num_proc=17): 100%|██████████| 12779/12779 [00:00<00:00, 25923.24 examples/s]


Dataset({
    features: ['id', 'table_csv', 'table_text', 'label', 'statement', 'table_caption', 'nlsep_query', 'semtab_query', 'nlsep_answ', 'nlsep_label', 'nlsep_answ_correct'],
    num_rows: 8974
})
70.22458721339699


Filter (num_proc=17): 100%|██████████| 8974/8974 [00:00<00:00, 18914.21 examples/s]

42.92980671414038


In [217]:
dd[0]

{'id': 0,
 'table_csv': '2-1570274-4.html.csv',
 'table_text': 'tournament#wins#top - 5#top - 10#top - 25#events#cuts made\nmasters tournament#0#1#2#4#4#4\nus open#0#2#3#4#6#5\nthe open championship#1#2#2#2#3#3\npga championship#0#0#1#2#5#4\ntotals#1#5#8#12#18#16\n',
 'label': 1,
 'statement': 'tournament that tony lema have participate in include the master tournament , the us open , the pga championship and the open championship',
 'table_caption': 'tony lema',
 'nlsep_query': 'tournament that tony lema have participate in include the master tournament , the us open , the pga championship and the open championship  col : tournament | wins | top - 5 | top - 10 | top - 25 | events | cuts made row 1 : masters tournament | 0 | 1 | 2 | 4 | 4 | 4 row 2 : us open | 0 | 2 | 3 | 4 | 6 | 5 row 3 : the open championship | 1 | 2 | 2 | 2 | 3 | 3 row 4 : pga championship | 0 | 0 | 1 | 2 | 5 | 4 row 5 : totals | 1 | 5 | 8 | 12 | 18 | 16',
 'semtab_query': 'tournament that tony lema have participate

In [216]:
process_data.filter(lambda x: True if x['nlsep_answ_correct']!='None' else False,num_proc=17)


Filter (num_proc=17): 100%|██████████| 12779/12779 [00:00<00:00, 26475.69 examples/s]


Dataset({
    features: ['id', 'table_csv', 'table_text', 'label', 'statement', 'table_caption', 'nlsep_query', 'semtab_query', 'nlsep_answ', 'nlsep_label', 'nlsep_answ_correct'],
    num_rows: 479
})

In [220]:
process_data.filter(lambda x: True if x['id']==905 else False,num_proc=17)[0]

Filter (num_proc=17): 100%|██████████| 12779/12779 [00:00<00:00, 26537.53 examples/s]


{'id': 905,
 'table_csv': '2-17231125-6.html.csv',
 'table_text': "place#player#country#score#to par\n1#curtis strange#united states#70 + 67 + 69 = 206#- 7\nt2#nick faldo#england#72 + 67 + 68 = 207#- 6\nt2#bob gilder#united states#68 + 69 + 70 = 207#- 6\nt2#scott simpson#united states#69 + 66 + 72 = 207#- 6\nt5#larry mize#united states#69 + 67 + 72 = 208#- 5\nt5#d a weibring#united states#71 + 69 + 68 = 208#- 5\n7#mark o'meara#united states#71 + 72 + 66 = 209#- 4\n8#fred couples#united states#72 + 67 + 71 = 210#- 3\n9#lanny wadkins#united states#70 + 71 + 70 = 211#- 2\n10#ken green#united states#72 + 70 + 70 = 212#- 1\n",
 'label': 1,
 'statement': 'player larry mize and d a be tie for 5th place',
 'table_caption': '1988 u.s. open (golf)',
 'nlsep_query': "player larry mize and d a be tie for 5th place  col : place | player | country | score | to par row 1 : 1 | curtis strange | united states | 70 + 67 + 69 = 206 | - 7 row 2 : t2 | nick faldo | england | 72 + 67 + 68 = 207 | - 6 row 3 

In [221]:
parse_panda_code('Here\'s the single-line pandas expression:\n\n```json\n"PANDA": "df.loc[df[\'score\'].isin(df.loc[df[\'place\']==5, \'score\']), \'player\'].str.contains(\'larry mize|d a weibring\').all()"\n```\n\nThis expression checks if all the players in the 5th place score are present in the score of \'larry mize\' and \'d a weibring\'.\n')

"df.loc[df['score'].isin(df.loc[df['place']==5, 'score']), 'player'].str.contains('larry mize|d a weibring').all()"

In [237]:
df = pd.read_csv(StringIO("place#player#country#score#to par\n1#curtis strange#united states#70 + 67 + 69 = 206#- 7\nt2#nick faldo#england#72 + 67 + 68 = 207#- 6\nt2#bob gilder#united states#68 + 69 + 70 = 207#- 6\nt2#scott simpson#united states#69 + 66 + 72 = 207#- 6\nt5#larry mize#united states#69 + 67 + 72 = 208#- 5\nt5#d a weibring#united states#71 + 69 + 68 = 208#- 5\n7#mark o'meara#united states#71 + 72 + 66 = 209#- 4\n8#fred couples#united states#72 + 67 + 71 = 210#- 3\n9#lanny wadkins#united states#70 + 71 + 70 = 211#- 2\n10#ken green#united states#72 + 70 + 70 = 212#- 1\n"), delimiter='#')


In [238]:
df

,place,player,country,score,to par
0,1,curtis strange,united states,70 + 67 + 69 = 206,- 7
1,t2,nick faldo,england,72 + 67 + 68 = 207,- 6
2,t2,bob gilder,united states,68 + 69 + 70 = 207,- 6
3,t2,scott simpson,united states,69 + 66 + 72 = 207,- 6
4,t5,larry mize,united states,69 + 67 + 72 = 208,- 5
5,t5,d a weibring,united states,71 + 69 + 68 = 208,- 5
6,7,mark o'meara,united states,71 + 72 + 66 = 209,- 4
7,8,fred couples,united states,72 + 67 + 71 = 210,- 3
8,9,lanny wadkins,united states,70 + 71 + 70 = 211,- 2
9,10,ken green,united states,72 + 70 + 70 = 212,- 1


In [239]:
df.dtypes

place      object
player     object
country    object
score      object
to par     object
dtype: object

In [230]:
for col in df.columns:
    try:
        df[col] = pd.to_numeric(df[col])
    except ValueError as e:
        print(col,e)
        # Если возникает ошибка, оставляем столбец как есть
        pass

place Unable to parse string "t2" at position 1
player Unable to parse string "curtis strange" at position 0
country Unable to parse string "united states" at position 0
score Unable to parse string "70 + 67 + 69 = 206" at position 0
to par Unable to parse string "- 7" at position 0


In [235]:
int('-7')

-7

In [236]:
eval("df.loc[df['score'].isin(df.loc[df['place']==5, 'score']), 'player'].str.contains('larry mize|d a weibring').all()")

np.True_

In [334]:
data = load_from_disk('tab_fact_test_xml_nlsep_new_correct_modify/')
print(data)
process_data = data.filter(lambda x : x['nlsep_answ_correct'] != 'None',num_proc=17)
print(process_data)
dd = process_data.filter(lambda x: True if x['nlsep_label']!='None' else False,num_proc=17)
print(dd.shape[0]/data.shape[0]*100)
print(dd.shape[0]/process_data.shape[0]*100)
true = dd.filter(lambda x : True if x['nlsep_label']==str(bool(x['label'])) else False,num_proc=17)
print(true.shape[0]/process_data.shape[0]*100)
full = data.filter(lambda x : True if x['nlsep_label']==str(bool(x['label'])) else False,num_proc=17)
print(full.shape[0]/data.shape[0]*100)


Dataset({
    features: ['id', 'table_csv', 'table_text', 'label', 'statement', 'table_caption', 'nlsep_query', 'semtab_query', 'nlsep_answ', 'nlsep_label', 'nlsep_answ_correct'],
    num_rows: 12779
})
Dataset({
    features: ['id', 'table_csv', 'table_text', 'label', 'statement', 'table_caption', 'nlsep_query', 'semtab_query', 'nlsep_answ', 'nlsep_label', 'nlsep_answ_correct'],
    num_rows: 2798
})
21.89529697159402
100.0
57.255182273052185


Filter (num_proc=17): 100%|██████████| 12779/12779 [00:00<00:00, 27316.22 examples/s]

55.450348227560845


In [329]:
true = dd.filter(lambda x : True if x['nlsep_label']==str(bool(x['label'])) else False,num_proc=17)


Dataset({
    features: ['id', 'table_csv', 'table_text', 'label', 'statement', 'table_caption', 'nlsep_query', 'semtab_query', 'nlsep_answ', 'nlsep_label', 'nlsep_answ_correct'],
    num_rows: 2798
})

In [335]:
data = load_from_disk('tab_fact_test_xml_semtab_new_correct_modify/')
print(data)
process_data = data.filter(lambda x : x['semtab_answ_correct'] != 'None',num_proc=17)
print(process_data)
dd = process_data.filter(lambda x: True if x['semtab_label']!='None' else False,num_proc=17)
print(dd.shape[0]/data.shape[0]*100)
print(dd.shape[0]/process_data.shape[0]*100)
true = dd.filter(lambda x : True if x['semtab_label']==str(bool(x['label'])) else False,num_proc=17)
print(true.shape[0]/process_data.shape[0]*100)
full = data.filter(lambda x : True if x['semtab_label']==str(bool(x['label'])) else False,num_proc=17)
print(full.shape[0]/data.shape[0]*100)


Dataset({
    features: ['id', 'table_csv', 'table_text', 'label', 'statement', 'table_caption', 'nlsep_query', 'semtab_query', 'semtab_answ', 'semtab_label', 'semtab_answ_correct'],
    num_rows: 12779
})


Filter (num_proc=17): 100%|██████████| 12779/12779 [00:00<00:00, 25449.74 examples/s]


Dataset({
    features: ['id', 'table_csv', 'table_text', 'label', 'statement', 'table_caption', 'nlsep_query', 'semtab_query', 'semtab_answ', 'semtab_label', 'semtab_answ_correct'],
    num_rows: 2727
})


Filter (num_proc=17): 100%|██████████| 2727/2727 [00:00<00:00, 5528.78 examples/s]


21.339697941935988
100.0


Filter (num_proc=17): 100%|██████████| 2727/2727 [00:00<00:00, 5716.06 examples/s]


54.49211587825449


Filter (num_proc=17): 100%|██████████| 12779/12779 [00:00<00:00, 25739.32 examples/s]

51.85069254245246


In [327]:
df_info

'tournament    object\nwins           int64\ntop - 5        int64\ntop - 10       int64\ntop - 25       int64\nevents         int64\ncuts made      int64'

In [336]:
data = load_from_disk('tab_fact_test_xml_semtab_new_correct_modify_add_info/')
print(data)
process_data = data.filter(lambda x : x['semtab_answ_correct'] != 'None',num_proc=17)
print(process_data)
dd = process_data.filter(lambda x: True if x['semtab_label']!='None' else False,num_proc=17)
print(dd.shape[0]/data.shape[0]*100)
print(dd.shape[0]/process_data.shape[0]*100)
true = dd.filter(lambda x : True if x['semtab_label']==str(bool(x['label'])) else False,num_proc=17)
print(true.shape[0]/process_data.shape[0]*100)
full = data.filter(lambda x : True if x['semtab_label']==str(bool(x['label'])) else False,num_proc=17)
print(full.shape[0]/data.shape[0]*100)

Dataset({
    features: ['id', 'table_csv', 'table_text', 'label', 'statement', 'table_caption', 'nlsep_query', 'semtab_query', 'semtab_answ', 'semtab_label', 'semtab_answ_correct'],
    num_rows: 12779
})


Filter (num_proc=17): 100%|██████████| 12779/12779 [00:00<00:00, 25729.94 examples/s]


Dataset({
    features: ['id', 'table_csv', 'table_text', 'label', 'statement', 'table_caption', 'nlsep_query', 'semtab_query', 'semtab_answ', 'semtab_label', 'semtab_answ_correct'],
    num_rows: 2876
})


Filter (num_proc=17): 100%|██████████| 2876/2876 [00:00<00:00, 5600.28 examples/s]


22.50567337037327
100.0


Filter (num_proc=17): 100%|██████████| 2876/2876 [00:00<00:00, 5845.32 examples/s]


54.24200278164116


Filter (num_proc=17): 100%|██████████| 12779/12779 [00:00<00:00, 26336.59 examples/s]

52.81320917129666


In [337]:
data = load_from_disk('tab_fact_test_xml_nlsep_new_correct_modify_noadd_info/')
print(data)
process_data = data.filter(lambda x : x['nlsep_answ_correct'] != 'None',num_proc=17)
print(process_data)
dd = process_data.filter(lambda x: True if x['nlsep_label']!='None' else False,num_proc=17)
print(dd.shape[0]/data.shape[0]*100)
print(dd.shape[0]/process_data.shape[0]*100)
true = dd.filter(lambda x : True if x['nlsep_label']==str(bool(x['label'])) else False,num_proc=17)
print(true.shape[0]/process_data.shape[0]*100)
full = data.filter(lambda x : True if x['nlsep_label']==str(bool(x['label'])) else False,num_proc=17)
print(full.shape[0]/data.shape[0]*100)

Dataset({
    features: ['id', 'table_csv', 'table_text', 'label', 'statement', 'table_caption', 'nlsep_query', 'semtab_query', 'nlsep_answ', 'nlsep_label', 'nlsep_answ_correct'],
    num_rows: 12779
})


Filter (num_proc=17): 100%|██████████| 12779/12779 [00:00<00:00, 25537.02 examples/s]


Dataset({
    features: ['id', 'table_csv', 'table_text', 'label', 'statement', 'table_caption', 'nlsep_query', 'semtab_query', 'nlsep_answ', 'nlsep_label', 'nlsep_answ_correct'],
    num_rows: 2747
})


Filter (num_proc=17): 100%|██████████| 2747/2747 [00:00<00:00, 5446.84 examples/s]


21.496204710853746
100.0


Filter (num_proc=17): 100%|██████████| 2747/2747 [00:00<00:00, 5683.92 examples/s]


58.79140880961048


Filter (num_proc=17): 100%|██████████| 12779/12779 [00:00<00:00, 26767.75 examples/s]

54.229595430002355


In [340]:
print(df.to_string())

              tournament  wins  top - 5  top - 10  top - 25  events  cuts made
0     masters tournament     0        1         2         4       4          4
1                us open     0        2         3         4       6          5
2  the open championship     1        2         2         2       3          3
3       pga championship     0        0         1         2       5          4
4                 totals     1        5         8        12      18         16


In [341]:
data

Dataset({
    features: ['id', 'table_csv', 'table_text', 'label', 'statement', 'table_caption', 'nlsep_query', 'semtab_query', 'nlsep_answ', 'nlsep_label', 'nlsep_answ_correct'],
    num_rows: 12779
})

In [342]:
data[0]['statement']

'tony lema be in the top 5 for the master tournament , the us open , and the open championship'

In [343]:
import yaml
def load_config(config_file_path):
    with open(config_file_path, 'r') as stream:
        try:
            # Use safe_load for security
            config = yaml.safe_load(stream)
            return config
        except yaml.YAMLError as exc:
            print(exc)
            

In [344]:
    config_data = load_config('convert_config.yaml')

In [347]:

config_data.keys()


dict_keys(['semtab_xml_elements'])

In [349]:
type(config_data['semtab_xml_elements']['include_data_types'])

bool

In [356]:
data = load_from_disk('tab_fact_test_universal2')

In [357]:
data

Dataset({
    features: ['id', 'table_csv', 'table_text', 'label', 'statement', 'table_caption', 'semtab_html_elements_semantic_datatype_exampples_description_top1_tresh50', 'semtab_xml_attributes_semantic_datatype_exampples_description_top1_tresh50', 'semtab_xml_elements_semantic_datatype_exampples_description_top1_tresh50', 'semtab_xml_elements_datatype_exampples_description_top1_tresh50', 'semtab_xml_elements_exampples_description_top1_tresh50', 'semtab_xml_elements_description_top1_tresh50'],
    num_rows: 12779
})

In [358]:
data[50]['semtab_html_elements_semantic_datatype_exampples_description_top1_tresh50']

'<table><caption>1989 u.s. open (golf)</caption><thead><tr><th SEMANTIC_TYPE="rank - 0.99" DATA_TYPE="&quot;str&quot;" HAS_NONE="0" EXAMPLES="[&quot;1&quot;, &quot;t6&quot;, &quot;t9&quot;]">place</th><th SEMANTIC_TYPE="name - 1.0" DATA_TYPE="&quot;str&quot;" HAS_NONE="0" EXAMPLES="[&quot;tom kite&quot;, &quot;brian claar&quot;, &quot;mark mccumber&quot;]">player</th><th SEMANTIC_TYPE="nationality - 1.0" DATA_TYPE="&quot;str&quot;" HAS_NONE="0" EXAMPLES="[&quot;united states&quot;, &quot;united states&quot;, &quot;spain&quot;]">country</th><th SEMANTIC_TYPE="weight - 0.91" DATA_TYPE="&quot;str&quot;" HAS_NONE="0" EXAMPLES="[&quot;70 + 68 + 73 + 68 = 279&quot;, &quot;71 + 70 + 71 + 70 = 282&quot;, &quot;70 + 71 + 68 + 72 = 281&quot;]">score</th><th SEMANTIC_TYPE="weight - 0.64" DATA_TYPE="&quot;str&quot;" HAS_NONE="0" EXAMPLES="[&quot;+ 1&quot;, &quot;+ 1&quot;, &quot;- 2&quot;]">to par</th><th SEMANTIC_TYPE="year - 0.97" DATA_TYPE="&quot;int&quot;" HAS_NONE="0" EXAMPLES="[200000, 28220

In [360]:
data = load_from_disk('tab_fact_test_xml_space_new_correct_modify_noadd_info/')
print(data)
process_data = data.filter(lambda x : x['space_answ_correct'] != 'None',num_proc=17)
print(process_data)
dd = process_data.filter(lambda x: True if x['space_label']!='None' else False,num_proc=17)
print(dd.shape[0]/data.shape[0]*100)
print(dd.shape[0]/process_data.shape[0]*100)
true = dd.filter(lambda x : True if x['space_label']==str(bool(x['label'])) else False,num_proc=17)
print(true.shape[0]/process_data.shape[0]*100)
full = data.filter(lambda x : True if x['space_label']==str(bool(x['label'])) else False,num_proc=17)
print(full.shape[0]/data.shape[0]*100)

Dataset({
    features: ['id', 'table_csv', 'table_text', 'label', 'statement', 'table_caption', 'nlsep_query', 'semtab_query', 'space_answ', 'space_label', 'space_answ_correct'],
    num_rows: 12779
})


Filter (num_proc=17): 100%|██████████| 12779/12779 [00:00<00:00, 25156.65 examples/s]


Dataset({
    features: ['id', 'table_csv', 'table_text', 'label', 'statement', 'table_caption', 'nlsep_query', 'semtab_query', 'space_answ', 'space_label', 'space_answ_correct'],
    num_rows: 3026
})


Filter (num_proc=17): 100%|██████████| 3026/3026 [00:00<00:00, 5870.48 examples/s]


23.679474137256438
100.0


Filter (num_proc=17): 100%|██████████| 3026/3026 [00:00<00:00, 5859.72 examples/s]


56.2128222075347


Filter (num_proc=17): 100%|██████████| 12779/12779 [00:00<00:00, 24946.84 examples/s]

53.28272947804993


In [361]:
tab_fact_test_semtab_html_elements_semantic_datatype_exampples_description_top1_tresh50_new_correct_modify_noadd_info


NameError: name 'tab_fact_test_semtab_html_elements_semantic_datatype_exampples_description_top1_tresh50_new_correct_modify_noadd_info' is not defined

In [362]:
data = load_from_disk('tab_fact_test_semtab_html_elements_semantic_datatype_exampples_description_top1_tresh50_new_correct_modify_noadd_info/')
print(data)
process_data = data.filter(lambda x : x['semtab_html_elements_semantic_datatype_exampples_description_top1_tresh50_answ_correct'] != 'None',num_proc=17)
print(process_data)
dd = process_data.filter(lambda x: True if x['semtab_html_elements_semantic_datatype_exampples_description_top1_tresh50_label']!='None' else False,num_proc=17)
print(dd.shape[0]/data.shape[0]*100)
print(dd.shape[0]/process_data.shape[0]*100)
true = dd.filter(lambda x : True if x['semtab_html_elements_semantic_datatype_exampples_description_top1_tresh50_label']==str(bool(x['label'])) else False,num_proc=17)
print(true.shape[0]/process_data.shape[0]*100)
full = data.filter(lambda x : True if x['semtab_html_elements_semantic_datatype_exampples_description_top1_tresh50_label']==str(bool(x['label'])) else False,num_proc=17)
print(full.shape[0]/data.shape[0]*100)

Dataset({
    features: ['id', 'table_csv', 'table_text', 'label', 'statement', 'table_caption', 'semtab_html_elements_semantic_datatype_exampples_description_top1_tresh50', 'semtab_xml_attributes_semantic_datatype_exampples_description_top1_tresh50', 'semtab_xml_elements_semantic_datatype_exampples_description_top1_tresh50', 'semtab_xml_elements_datatype_exampples_description_top1_tresh50', 'semtab_xml_elements_exampples_description_top1_tresh50', 'semtab_xml_elements_description_top1_tresh50', 'semtab_html_elements_semantic_datatype_exampples_description_top1_tresh50_answ', 'semtab_html_elements_semantic_datatype_exampples_description_top1_tresh50_label', 'semtab_html_elements_semantic_datatype_exampples_description_top1_tresh50_answ_correct'],
    num_rows: 12779
})


Filter (num_proc=17): 100%|██████████| 12779/12779 [00:00<00:00, 25331.21 examples/s]


Dataset({
    features: ['id', 'table_csv', 'table_text', 'label', 'statement', 'table_caption', 'semtab_html_elements_semantic_datatype_exampples_description_top1_tresh50', 'semtab_xml_attributes_semantic_datatype_exampples_description_top1_tresh50', 'semtab_xml_elements_semantic_datatype_exampples_description_top1_tresh50', 'semtab_xml_elements_datatype_exampples_description_top1_tresh50', 'semtab_xml_elements_exampples_description_top1_tresh50', 'semtab_xml_elements_description_top1_tresh50', 'semtab_html_elements_semantic_datatype_exampples_description_top1_tresh50_answ', 'semtab_html_elements_semantic_datatype_exampples_description_top1_tresh50_label', 'semtab_html_elements_semantic_datatype_exampples_description_top1_tresh50_answ_correct'],
    num_rows: 2399
})


Filter (num_proc=17): 100%|██████████| 2399/2399 [00:00<00:00, 4543.86 examples/s]


18.772986931684795
100.0


Filter (num_proc=17): 100%|██████████| 2399/2399 [00:00<00:00, 4636.70 examples/s]


54.18924551896623


Filter (num_proc=17): 100%|██████████| 12779/12779 [00:00<00:00, 24993.23 examples/s]

53.86180452304562


| Method              | Accuracy |
|---------------------|:--------:|
| semtab xml elements |   51.85  |
| semtab html         |   53.86  |
| space               |   53.28  |
| nlsep               |   54.23  |

In [364]:
df.to_markdown()

'|    | tournament            |   wins |   top - 5 |   top - 10 |   top - 25 |   events |   cuts made |\n|---:|:----------------------|-------:|----------:|-----------:|-----------:|---------:|------------:|\n|  0 | masters tournament    |      0 |         1 |          2 |          4 |        4 |           4 |\n|  1 | us open               |      0 |         2 |          3 |          4 |        6 |           5 |\n|  2 | the open championship |      1 |         2 |          2 |          2 |        3 |           3 |\n|  3 | pga championship      |      0 |         0 |          1 |          2 |        5 |           4 |\n|  4 | totals                |      1 |         5 |          8 |         12 |       18 |          16 |'

In [ ]:
python abation_experiments.py --inputdata tab_fact_test_semtab_html_ablation_correcring --outputdata tab_fact_test_semtab_html_ablation_correcring --conf-file semtab_html_config_answer2.yaml --num_proc 16 > correct_tabfact_semtab_html_log2.txt

In [365]:
import torch


In [367]:
str(torch.device('cuda'))


'cuda'

In [371]:
data = load_from_disk('tab_fact_test_semtab_html_ablation/')

In [372]:
data


Dataset({
    features: ['id', 'table_csv', 'table_text', 'label', 'statement', 'table_caption', 'semtab_html_semantic_datatype_exampples3_description_top1_tresh50', 'semtab_html_datatype_exampples3_description_top1_tresh50', 'semtab_html_exampples3_description_top1_tresh50', 'semtab_html_description_top1_tresh50', 'semtab_html_top1_tresh50'],
    num_rows: 12779
})

In [374]:
data[0]['semtab_html_top1_tresh50']


'<table><thead><tr><th>tournament</th><th>wins</th><th>top - 5</th><th>top - 10</th><th>top - 25</th><th>events</th><th>cuts made</th></tr></thead></table>'

# Ablation study

In [376]:
from datasets import load_from_disk

In [381]:
data = load_from_disk("tab_fact_test_semtab_html_ablation_correcring_first_new")

In [382]:
data


Dataset({
    features: ['id', 'table_csv', 'table_text', 'label', 'statement', 'table_caption', 'semtab_html_semantic_datatype_exampples3_description_top1_tresh50', 'semtab_html_datatype_exampples3_description_top1_tresh50', 'semtab_html_exampples3_description_top1_tresh50', 'semtab_html_description_top1_tresh50', 'semtab_html_top1_tresh50', 'semtab_html_semantic_datatype_exampples3_description_top1_tresh50_answ', 'semtab_html_semantic_datatype_exampples3_description_top1_tresh50_label', 'semtab_html_semantic_datatype_exampples3_description_top1_tresh50_answ_correct', 'semtab_html_datatype_exampples3_description_top1_tresh50_answ', 'semtab_html_datatype_exampples3_description_top1_tresh50_label', 'semtab_html_datatype_exampples3_description_top1_tresh50_answ_correct', 'semtab_html_exampples3_description_top1_tresh50_answ', 'semtab_html_exampples3_description_top1_tresh50_label', 'semtab_html_exampples3_description_top1_tresh50_answ_correct', 'semtab_html_description_top1_tresh50_answ'

In [383]:
from utils.utils import load_config


In [384]:
config = load_config('semtab_html_config_answer2.yaml')

In [386]:
tests = list(config.keys())
tests

['semtab_html_semantic_datatype_exampples3_description_top1_tresh50',
 'semtab_html_datatype_exampples3_description_top1_tresh50',
 'semtab_html_exampples3_description_top1_tresh50',
 'semtab_html_description_top1_tresh50',
 'semtab_html_top1_tresh50']

In [391]:
acc_old = None
for test in tests:
    print(test)
    process_data = data.filter(lambda x : x[f'{test}_answ_correct'] != 'None',num_proc=17)
    dd = process_data.filter(lambda x: True if x[f'{test}_label']!='None' else False,num_proc=17)
    #print(dd.shape[0]/data.shape[0]*100)
    #print(dd.shape[0]/process_data.shape[0]*100)
    true = dd.filter(lambda x : True if x[f'{test}_label']==str(bool(x['label'])) else False,num_proc=17)
    #print(true.shape[0]/process_data.shape[0]*100)
    full = data.filter(lambda x : True if x[f'{test}_label']==str(bool(x['label'])) else False,num_proc=17)
    acc = full.shape[0]/data.shape[0]*100
    print(acc)
    if acc_old == None:
        acc_old = acc
    print('different',acc-acc_old)
    acc_old = acc

semtab_html_semantic_datatype_exampples3_description_top1_tresh50
53.673996400344315
different 0.0
semtab_html_datatype_exampples3_description_top1_tresh50
54.816495813443936
different 1.142499413099621
semtab_html_exampples3_description_top1_tresh50
54.11221535331403
different -0.7042804601299082
semtab_html_description_top1_tresh50
46.545113076140545
different -7.567102277173483
semtab_html_top1_tresh50
46.490335707019334
different -0.05477736912121145


In [392]:
data

Dataset({
    features: ['id', 'table_csv', 'table_text', 'label', 'statement', 'table_caption', 'semtab_html_semantic_datatype_exampples3_description_top1_tresh50', 'semtab_html_datatype_exampples3_description_top1_tresh50', 'semtab_html_exampples3_description_top1_tresh50', 'semtab_html_description_top1_tresh50', 'semtab_html_top1_tresh50', 'semtab_html_semantic_datatype_exampples3_description_top1_tresh50_answ', 'semtab_html_semantic_datatype_exampples3_description_top1_tresh50_label', 'semtab_html_semantic_datatype_exampples3_description_top1_tresh50_answ_correct', 'semtab_html_datatype_exampples3_description_top1_tresh50_answ', 'semtab_html_datatype_exampples3_description_top1_tresh50_label', 'semtab_html_datatype_exampples3_description_top1_tresh50_answ_correct', 'semtab_html_exampples3_description_top1_tresh50_answ', 'semtab_html_exampples3_description_top1_tresh50_label', 'semtab_html_exampples3_description_top1_tresh50_answ_correct', 'semtab_html_description_top1_tresh50_answ'

In [394]:
print(data[0]['table_text'])

tournament#wins#top - 5#top - 10#top - 25#events#cuts made
masters tournament#0#1#2#4#4#4
us open#0#2#3#4#6#5
the open championship#1#2#2#2#3#3
pga championship#0#0#1#2#5#4
totals#1#5#8#12#18#16



In [395]:
data2 = dataset.to_pandas()

In [396]:
data2


,id,statement,label,table_caption,table_text,pandas_code,pandas_eval,nlsep_query,semtab_query
0,0,haroldo be mention as a brazil scorer for 2 di...,1,1919 in brazilian football,date#result#score#brazil scorers#competition\n...,df['brazil scorers'].apply(lambda x: 'haroldo'...,True,haroldo be mention as a brazil scorer for 2 di...,haroldo be mention as a brazil scorer for 2 di...
1,1,4 of the 5 game be for the south american cham...,1,1919 in brazilian football,date#result#score#brazil scorers#competition\n...,(df['competition'].value_counts()['south ameri...,True,4 of the 5 game be for the south american cham...,4 of the 5 game be for the south american cham...
2,2,friedenreich be mention as a brazil scorer for...,1,1919 in brazilian football,date#result#score#brazil scorers#competition\n...,df['brazil scorers'].str.contains('friedenreic...,True,friedenreich be mention as a brazil scorer for...,friedenreich be mention as a brazil scorer for...
3,3,there be 2 different game where the highest sc...,1,1919 in brazilian football,date#result#score#brazil scorers#competition\n...,len(df[df['score'].str.extract(r'^(\d+) - \d+$...,True,there be 2 different game where the highest sc...,there be 2 different game where the highest sc...
4,4,4 of the 5 game be play in may 1919,1,1919 in brazilian football,date#result#score#brazil scorers#competition\n...,(df['date'].str.contains('may') & df['date'].s...,True,4 of the 5 game be play in may 1919 col : dat...,4 of the 5 game be play in may 1919 <TABLE><DE...
...,...,...,...,...,...,...,...,...,...
88111,92278,"jerraud power , 192 pound (87 kg) , be choose ...",1,2009 indianapolis colts season,round#choice#player#position#height#weight#col...,df[(df['player'] == 'jerraud powers') & (df['w...,True,"jerraud power , 192 pound (87 kg) , be choose ...","jerraud power , 192 pound (87 kg) , be choose ..."
88112,92279,terrence taylor play for auburn,0,2009 indianapolis colts season,round#choice#player#position#height#weight#col...,df[df['player'] == 'terrance taylor']['college...,False,terrence taylor play for auburn col : round |...,terrence taylor play for auburn <TABLE><DESCRI...
88113,92280,curtis painter 's position be quarterback and ...,0,2009 indianapolis colts season,round#choice#player#position#height#weight#col...,(df[df['player'] == 'curtis painter']['positio...,False,curtis painter 's position be quarterback and ...,curtis painter 's position be quarterback and ...
88114,92281,the player who weigh the most play in round 3,0,2009 indianapolis colts season,round#choice#player#position#height#weight#col...,(df.loc[df['weight'].str.extract('(\\d+)').ast...,False,the player who weigh the most play in round 3 ...,the player who weigh the most play in round 3 ...


In [397]:
data2['table_csv']


KeyError: 'table_csv'

In [401]:
data['table_csv']

Column(['2-1570274-4.html.csv', '2-1570274-4.html.csv', '2-1570274-4.html.csv', '2-1570274-4.html.csv', '2-1570274-4.html.csv'])

In [403]:
data['statement']

Column(['tony lema be in the top 5 for the master tournament , the us open , and the open championship', 'tournament that tony lema have participate in include the master tournament , the us open , the pga championship and the open championship', 'the only tournament that tony lema win in be the open championship', 'tony lema do not win in the us open', 'tony lema make it to the top 10 in the pga championship , but do not continue on'])

In [34]:
from datasets import load_from_disk,Dataset

In [2]:
data = load_from_disk('tab_fact_test_semtab_xml_ablation_new')

In [410]:
arr = [ i for i in data['table_csv']]
len(set(arr))

1695

In [5]:
data.filter(lambda x: x['semtab_xml_elements_top1_tresh50'] != 'None',num_proc=17)

Filter (num_proc=17): 100%|██████████| 12779/12779 [00:00<00:00, 28516.71 examples/s]


Dataset({
    features: ['id', 'table_csv', 'table_text', 'label', 'statement', 'table_caption', 'semtab_xml_attributes_semantic_datatype_exampples_description_top1_tresh50', 'semtab_xml_attributes_datatype_exampples_description_top1_tresh50', 'semtab_xml_attributes_exampples_description_top1_tresh50', 'semtab_xml_attributes_description_top1_tresh50', 'semtab_xml_attributes_top1_tresh50', 'semtab_xml_elements_semantic_datatype_exampples_description_top1_tresh50', 'semtab_xml_elements_datatype_exampples_description_top1_tresh50', 'semtab_xml_elements_exampples_description_top1_tresh50', 'semtab_xml_elements_description_top1_tresh50', 'semtab_xml_elements_top1_tresh50', 'semtab_xml_attributes_top1_tresh50_answ', 'semtab_xml_attributes_top1_tresh50_label', 'semtab_xml_attributes_top1_tresh50_answ_correct', 'semtab_xml_elements_semantic_datatype_exampples_description_top1_tresh50_answ', 'semtab_xml_elements_semantic_datatype_exampples_description_top1_tresh50_label', 'semtab_xml_elements_s

In [6]:
data[3434]['semtab_xml_elements_top1_tresh50']

'<TABLE><HEADER><NAME>res</NAME></HEADER><HEADER><NAME>record</NAME></HEADER><HEADER><NAME>opponent</NAME></HEADER><HEADER><NAME>method</NAME></HEADER><HEADER><NAME>event</NAME></HEADER><HEADER><NAME>round</NAME></HEADER><HEADER><NAME>time</NAME></HEADER><HEADER><NAME>location</NAME></HEADER></TABLE>'

In [25]:
data1 = load_from_disk("tab_fact_test_semtab_xml_ablation_new")
data_continue = load_from_disk("tab_fact_test_semtab_xml_ablation_new_continue")

In [18]:
from utils.utils import load_config


In [38]:
config = load_config('semtab_xml_config_answer2.yaml')
conf_continue = load_config('semtab_xml_config_answer2_continue.yaml')

In [20]:
tests_continue = list(conf_continue.keys())
tests_continue

['semtab_xml_attributes_top1_tresh50',
 'semtab_xml_elements_semantic_datatype_exampples_description_top1_tresh50',
 'semtab_xml_elements_datatype_exampples_description_top1_tresh50',
 'semtab_xml_elements_exampples_description_top1_tresh50',
 'semtab_xml_elements_description_top1_tresh50',
 'semtab_xml_elements_top1_tresh50']

In [39]:
tests = list(config.keys())
tests

['semtab_xml_attributes_semantic_datatype_exampples_description_top1_tresh50',
 'semtab_xml_attributes_datatype_exampples_description_top1_tresh50',
 'semtab_xml_attributes_exampples_description_top1_tresh50',
 'semtab_xml_attributes_description_top1_tresh50',
 'semtab_xml_attributes_top1_tresh50',
 'semtab_xml_elements_semantic_datatype_exampples_description_top1_tresh50',
 'semtab_xml_elements_datatype_exampples_description_top1_tresh50',
 'semtab_xml_elements_exampples_description_top1_tresh50',
 'semtab_xml_elements_description_top1_tresh50',
 'semtab_xml_elements_top1_tresh50']

In [31]:
from tqdm import tqdm
# 'semtab_xml_elements_top1_tresh50_answ', 'semtab_xml_elements_top1_tresh50_label', 'semtab_xml_elements_top1_tresh50_answ_correct'
ddata1 = data1.to_dict()
for expr in tqdm(tests_continue):
    print(expr)
    ddata1[f'{expr}_answ'] = data_continue[f'{expr}_answ']
    ddata1[f'{expr}_label'] = data_continue[f'{expr}_label']
    ddata1[f'{expr}_answ_correct'] = data_continue[f'{expr}_answ_correct']
   

100%|██████████| 6/6 [00:00<00:00, 141.79it/s]

semtab_xml_attributes_top1_tresh50
semtab_xml_elements_semantic_datatype_exampples_description_top1_tresh50
semtab_xml_elements_datatype_exampples_description_top1_tresh50
semtab_xml_elements_exampples_description_top1_tresh50
semtab_xml_elements_description_top1_tresh50
semtab_xml_elements_top1_tresh50


In [35]:
data1 = Dataset.from_dict(ddata1)

In [37]:
data1.save_to_disk('tab_fact_test_semtab_xml_ablation_new')

Saving the dataset (1/1 shards): 100%|██████████| 12779/12779 [00:00<00:00, 39481.09 examples/s]


In [40]:
data = data1

In [41]:
acc_old = None
for test in tests:
    print(test)
    process_data = data.filter(lambda x : x[f'{test}_answ_correct'] != 'None',num_proc=17)
    dd = process_data.filter(lambda x: True if x[f'{test}_label']!='None' else False,num_proc=17)
    #print(dd.shape[0]/data.shape[0]*100)
    #print(dd.shape[0]/process_data.shape[0]*100)
    true = dd.filter(lambda x : True if x[f'{test}_label']==str(bool(x['label'])) else False,num_proc=17)
    #print(true.shape[0]/process_data.shape[0]*100)
    full = data.filter(lambda x : True if x[f'{test}_label']==str(bool(x['label'])) else False,num_proc=17)
    acc = full.shape[0]/data.shape[0]*100
    print(acc)
    if acc_old == None:
        acc_old = acc
    print('different',acc-acc_old)
    acc_old = acc

semtab_xml_attributes_semantic_datatype_exampples_description_top1_tresh50


Filter (num_proc=17): 100%|██████████| 12779/12779 [00:16<00:00, 768.34 examples/s]


54.1982940762188
different 0.0
semtab_xml_attributes_datatype_exampples_description_top1_tresh50


Filter (num_proc=17): 100%|██████████| 12779/12779 [00:16<00:00, 765.33 examples/s]


55.176461381954766
different 0.9781673057359654
semtab_xml_attributes_exampples_description_top1_tresh50


...

51.529853666171064
different -0.8842632443853233
semtab_xml_elements_description_top1_tresh50


Filter (num_proc=17): 100%|██████████| 12779/12779 [00:16<00:00, 754.09 examples/s]


43.13326551373347
different -8.396588152437594
semtab_xml_elements_top1_tresh50


Filter (num_proc=17): 100%|██████████| 12779/12779 [00:17<00:00, 750.14 examples/s]

43.61843649737851
different 0.4851709836450411
